## 1. My rule and its reason codes

### Signal checks

**Signal 1 — Volume:** previous-30-day impressions. This is linked to the quick-win/volume idea.

**Verdict: CONFIRMED** — the bucket table below shows measurable content volume bands.

**Signal 2 — CTR vs position:** previous-30-day CTR compared with average search position. This is linked to the CTR-fix logic.

**Verdict: MIXED** — CTR varies across position bands, but position alone does not determine CTR. It is therefore a supporting signal, not a standalone diagnosis.

### Rule

Rank content items by a **0–100 review score** using only February observations:
- 0–40 points for observed volume.
- 0–40 points for CTR relative to the item's observed position bucket.
- 0–20 points for position volatility.

**One reason code:** `REFRESH_REVIEW` for scores >= 60; `MONITOR` otherwise.

**Action labels:** `Review for refresh` or `Monitor`.

This is a prioritization baseline, not a claim that a page will decline.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy
import os,getpass,duckdb,pandas as pd,numpy as np
HF_TOKEN=os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN=userdata.get('HF_TOKEN')
    except Exception: pass
HF_TOKEN=HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
assert HF_TOKEN
con=duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL='hf://datasets/FlyRank/internship-warehouse'
DAILY=f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
print('Connected to FlyRank warehouse.')

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse.


In [2]:
df=con.sql(f"""
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions_30d,
       SUM(gsc_clicks) AS clicks_30d,
       CASE WHEN SUM(gsc_impressions)>0 THEN SUM(gsc_clicks)*1.0/SUM(gsc_impressions) ELSE NULL END AS ctr_30d,
       AVG(gsc_avg_position) AS avg_position_30d,
       STDDEV_SAMP(gsc_avg_position) AS position_volatility
FROM {DAILY}
WHERE report_date >= DATE '2026-02-01' AND report_date < DATE '2026-03-01'
GROUP BY 1,2
HAVING SUM(gsc_impressions) >= 20
""").df()
print(f'Content items available for the baseline: {len(df):,}')
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items available for the baseline: 108,654


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,ctr_30d,avg_position_30d,position_volatility
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,29.609070,7.831937
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,17.806923,5.236842
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,7.852945,3.643439
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,7.123694,5.363122
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,7.965842,6.686050


In [3]:
df['volume_bucket']=pd.cut(df['impressions_30d'],bins=[0,100,500,2000,np.inf],labels=['20-100','101-500','501-2000','2000+'],include_lowest=True)
volume_table=df.groupby('volume_bucket',observed=False).size().reset_index(name='n')
print('Signal 1 — Volume bucket table (n):'); display(volume_table)
assert volume_table['n'].sum()==len(df)

df['position_bucket']=pd.cut(df['avg_position_30d'],bins=[0,3,10,20,50,np.inf],labels=['1-3','4-10','11-20','21-50','50+'],include_lowest=True)
ctr_table=df.groupby('position_bucket',observed=False).agg(n=('ctr_30d','size'),median_ctr=('ctr_30d','median'),mean_ctr=('ctr_30d','mean')).reset_index()
print('Signal 2 — CTR by average-position bucket:'); display(ctr_table)

Signal 1 — Volume bucket table (n):


,volume_bucket,n
0,20-100,28530
1,101-500,33022
2,501-2000,26066
3,2000+,21036


Signal 2 — CTR by average-position bucket:


,position_bucket,n,median_ctr,mean_ctr
0,1-3,12719,0.001678,0.003227
1,4-10,53331,0.001110,0.003411
2,11-20,23777,0.000000,0.002373
3,21-50,16097,0.000000,0.001540
4,50+,2730,0.000000,0.000821


## 2. Build the ranked queue (writes the CSV)

The score is transparent and uses no future window, outcome label, or product flag. The queue is written to `work/outputs/baseline_action_score.csv`.

In [4]:
df['volume_score']=np.select([df.impressions_30d>=2000,df.impressions_30d>=500,df.impressions_30d>=100],[40,30,20],default=10)
pos_median=df.groupby('position_bucket',observed=False)['ctr_30d'].transform('median')
df['ctr_gap']=df['ctr_30d']-pos_median
df['ctr_score']=np.select([df.ctr_gap<=-0.005,df.ctr_gap<=-0.002,df.ctr_gap<0],[40,30,20],default=5)
vol_median=df['position_volatility'].median()
df['volatility_score']=np.select([df.position_volatility>=vol_median*2,df.position_volatility>=vol_median],[20,10],default=0)
df['score']=(df.volume_score+df.ctr_score+df.volatility_score).clip(0,100)
df['reason_code']=np.where(df.score>=60,'REFRESH_REVIEW','MONITOR')
df['action']=np.where(df.score>=60,'Review for refresh','Monitor')
df=df.sort_values(['score','impressions_30d'],ascending=[False,False]).reset_index(drop=True)
df['rank']=np.arange(1,len(df)+1)
output_cols=['rank','client_hash_id','content_hash_id','score','reason_code','action','impressions_30d','ctr_30d','avg_position_30d','position_volatility']
queue=df[output_cols].copy()
import pathlib
pathlib.Path('work/outputs').mkdir(parents=True,exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv',index=False)
print(f'Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv')
display(queue.head(10))

Wrote 108,654 rows to work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions_30d,ctr_30d,avg_position_30d,position_volatility
0,1,client_73cda7b4e4f265ea,content_c9f840183215651b,80,REFRESH_REVIEW,Review for refresh,125035.0,0.000000,9.366950,10.842532
1,2,client_9958f0a7ae1df715,content_179a20261f5ff964,80,REFRESH_REVIEW,Review for refresh,19060.0,0.000315,5.972741,10.436697
2,3,client_62f4a7e64f5e0096,content_4e576d3e62c1de8e,80,REFRESH_REVIEW,Review for refresh,8580.0,0.000000,3.200057,14.452360
3,4,client_62f4a7e64f5e0096,content_ac671311da8857d9,80,REFRESH_REVIEW,Review for refresh,5682.0,0.000176,3.128459,11.050494
4,5,client_62f4a7e64f5e0096,content_69a2fe59340878dc,80,REFRESH_REVIEW,Review for refresh,5486.0,0.000000,6.792724,12.622632
5,6,client_e547b89c05043229,content_feb20a281a923fc6,80,REFRESH_REVIEW,Review for refresh,4399.0,0.000227,9.864529,10.972088
6,7,client_23a62021009f63c4,content_7aa1533b112c76d7,80,REFRESH_REVIEW,Review for refresh,4097.0,0.000244,3.200290,9.068519
7,8,client_23a62021009f63c4,content_4daf2e9905721747,80,REFRESH_REVIEW,Review for refresh,3261.0,0.000307,9.567662,12.176878
8,9,client_73cda7b4e4f265ea,content_c421f827f5d0318c,80,REFRESH_REVIEW,Review for refresh,3126.0,0.000320,9.671087,8.999844
9,10,client_23a62021009f63c4,content_9e5b9b8c4d5ad594,80,REFRESH_REVIEW,Review for refresh,3025.0,0.000661,7.947032,11.011989


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top20=df.head(20).copy()
top20['confidence_note']=np.where(top20.score>=80,'Higher baseline priority','Moderate baseline priority')
top20['what_would_make_it_wrong']=np.select([top20.ctr_gap<=-0.005,top20.position_volatility>=vol_median*2],['CTR may reflect measurement or query-mix effects rather than a content issue','Position volatility may reflect temporary search movement rather than staleness'],default='The rule is a screen; manual content and SERP review could show no refresh opportunity.')
review_cols=['rank','action','reason_code','score','confidence_note','what_would_make_it_wrong']
display(top20[review_cols].to_string(index=False))
print('Top-20 rows reviewed:',len(top20))
assert len(top20)==min(20,len(df))

' rank             action    reason_code  score          confidence_note                                                        what_would_make_it_wrong\n    1 Review for refresh REFRESH_REVIEW     80 Higher baseline priority Position volatility may reflect temporary search movement rather than staleness\n    2 Review for refresh REFRESH_REVIEW     80 Higher baseline priority Position volatility may reflect temporary search movement rather than staleness\n    3 Review for refresh REFRESH_REVIEW     80 Higher baseline priority Position volatility may reflect temporary search movement rather than staleness\n    4 Review for refresh REFRESH_REVIEW     80 Higher baseline priority Position volatility may reflect temporary search movement rather than staleness\n    5 Review for refresh REFRESH_REVIEW     80 Higher baseline priority Position volatility may reflect temporary search movement rather than staleness\n    6 Review for refresh REFRESH_REVIEW     80 Higher baseline priority Position 

Top-20 rows reviewed: 20


## 4. Weak picks + leakage check

A high score can still be wrong. CTR differences may reflect query mix or SERP effects, while position volatility can be temporary. The baseline creates a review queue rather than an automatic content-change decision.

All scoring inputs come from February 2026. No future-window outcome, label, product flag, client name, URL, or private query text is used.

In [6]:
score_inputs=['impressions_30d','ctr_30d','avg_position_30d','position_volatility']
print('Score inputs:',score_inputs)
leak_terms=['future','label','target','outcome','next','march','april']
leak_hits=[c for c in score_inputs if any(k in c.lower() for k in leak_terms)]
print('Future/label fields in score inputs:',leak_hits)
assert not leak_hits
print('Leakage check: PASSED')
weak=top20.sort_values('score').head(5)
print('Five weakest top-20 picks for skeptic review:'); display(weak[['rank','score','action','ctr_30d','avg_position_30d','position_volatility']])

Score inputs: ['impressions_30d', 'ctr_30d', 'avg_position_30d', 'position_volatility']
Future/label fields in score inputs: []
Leakage check: PASSED
Five weakest top-20 picks for skeptic review:


,rank,score,action,ctr_30d,avg_position_30d,position_volatility
0,1,80,Review for refresh,0.000000,9.366950,10.842532
1,2,80,Review for refresh,0.000315,5.972741,10.436697
2,3,80,Review for refresh,0.000000,3.200057,14.452360
3,4,80,Review for refresh,0.000176,3.128459,11.050494
4,5,80,Review for refresh,0.000000,6.792724,12.622632


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.